# Imports

In [1]:
from xarm.wrapper import XArmAPI
import numpy as np

SDK_VERSION: 1.18.5


# Connecting to XArm

In [2]:
ARM_IP = '192.168.1.205'  # change this to xArm IP
arm = XArmAPI(ARM_IP, do_not_open=True)
arm.connect()

arm.clean_warn()
arm.clean_error()
arm.motion_enable(enable=True)

ROBOT_IP: 192.168.1.205, VERSION: v2.7.0, PROTOCOL: V1, DETAIL: 6,6,XI1304,AC1303,v2.7.0, TYPE1300: [1, 1]
change protocol identifier to 3
[motion_enable], xArm is not ready to move


0

# Arm Setup

In [3]:
arm.motion_enable(enable=True) #enabling motion on arm
arm.set_world_offset([0,0,8,0,0,0]) #8 mm offset due to mounting plate
arm.set_tcp_offset([119.569, 0, 12.6, 0, 0, -1]) #tool coordinates offset to tip of tool
arm.set_state(state=0)
arm.set_tcp_load(weight = 0.4997 + 0.048, center_of_gravity=[41,0,18]) #weight of tool in kg. Center of gravity in mm
arm.set_mode(0)  #Mode 0, positional control
arm.set_state(state=0)

[motion_enable], xArm is not ready to move
[set_state], xArm is ready to move


0

In [5]:
def zprobe(count):
    """
    Detect height of object by probing in the z direction.

    count: number of probes
    """
    readings = np.zeros(count)
    for i in range(count):
        while True:
            if arm.get_tgpio_digital(ionum = 1)[1] == 1:
                readings[i] = arm.get_position()[1][2]
                arm.set_tool_position(z = -5, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(z = 0.5, speed = 5, wait = True)
    return np.mean(readings), np.std(readings), readings


def zprobeRel(count):
    """
    Detect height of object relative to tool by probing in the z direction.

    count: number of probes
    """
    readings = np.zeros(count)

    for i in range(count):
        distance = 0
        while True:
            if arm.get_tgpio_digital(ionum = 1)[1] == 1:
                readings[i] = distance
                arm.set_tool_position(z = -1 * distance, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(z = 0.5, speed = 5, wait = True)
                distance += 0.5
    return np.mean(readings), np.std(readings), readings

def xprobe(count):
    """
    Detect position of object by probing in the tool's x direction; move the arm closer to the object in x dir

    count: number of probes
    """
    readings = np.zeros(count)
    for i in range(count):
        while True:
            if arm.get_tgpio_digital(ionum = 0)[1] == 1:
                readings[i] = arm.get_position()[1][1]
                arm.set_tool_position(x = -5, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(x = 0.5, speed = 5, wait = True)
    return np.mean(readings), np.std(readings), readings

def xprobeRel(count):
    """
    Detect position of object relative to tool by probing in the tool's x direction.

    count: number of probes
    """
    readings = np.zeros(count)
    for i in range(count):
        distance = 0
        while True:
            if arm.get_tgpio_digital(ionum = 0)[1] == 1:
                readings[i] = distance
                arm.set_tool_position(x = -1 * distance, speed = 50, wait = True)
                break
            else:
                arm.set_tool_position(x = 0.5, speed = 5, wait = True)
                distance += 0.5
    return np.mean(readings), np.std(readings), readings

def findYaw():
    """
    Adjust yaw of end effector based on position of plate.
    """
    xprobe(1)
    arm.set_tool_position(y = 5, speed = 50, wait = True)
    p1 = xprobeRel(1)[0]
    arm.set_tool_position(y = -10, speed = 50, wait = True)
    p2 = xprobeRel(1)[0]
    arm.set_tool_position(x = -5, y = 5, speed = 50, wait = True)
    arm.set_tool_position(yaw = -180*np.arctan((p1 - p2)/10)/np.pi, wait = True)
    arm.set_tool_position(y = 80, speed = 100, wait = True)
    p3 = xprobeRel(1)[0]
    arm.set_tool_position(y = -160, speed = 100, wait = True)
    p4 = xprobeRel(1)[0]
    
    arm.set_tool_position(x = -10, y = 80, speed = 100, wait = True)
    arm.set_tool_position(yaw = -180*np.arctan((p3 - p4)/160)/np.pi, wait = True)
    return -180*np.arctan((p1 - p2)/10)/np.pi, -180*np.arctan((p3 - p4)/160)/np.pi

def findpoint():
    """
    Detect raised point on buildplate.

    Starting from the current y position, move along +tool-y in 0.5 mm steps.
    At each step, probe in tool-x direction and compare the contact distance
    with the previous probe. If abs(x1 - x2) > 1.5 mm, assume a raised point
    or edge feature is detected. Then move y = -40 mm to the final reference pose.
    """

    STEP_Y = 0.5
    DETECT_THRESHOLD = 0.8
    MAX_TRAVEL_Y = 50

    # First align to the edge, then get the initial probe distance
    xprobe(1)
    x1 = xprobeRel(1)[0]

    traveled_y = 0

    while traveled_y < MAX_TRAVEL_Y:
        # Move forward along tool-y
        arm.set_tool_position(y=STEP_Y, speed=50, wait=True)
        traveled_y += STEP_Y

        # Measure new x probe distance
        x2 = xprobeRel(1)[0]

        print("traveled_y:", traveled_y, "x1:", x1, "x2:", x2, "delta:", x1 - x2)

        # Detect sudden change in edge distance
        if abs(x1 - x2) > DETECT_THRESHOLD:
            print("Detected raised point / edge feature")
            print("detected at relative y:", traveled_y)

            # Move from detected feature to final reference pose
            arm.set_tool_position(y=-40, speed=50, wait=True)
            return

        # Update previous distance
        x1 = x2

    print("error: raised point not found within", MAX_TRAVEL_Y, "mm")

    
def zlevel():
    """
    Probe build plate in the z direction to determine roll and pitch of end effector.
    """
    xd = 60
    yd = 40
    p1 = np.array([0, 0, 0])
    p2 = np.array([xd, yd, 0])
    p3 = np.array([xd, -yd, 0])
    zprobe(1)
    arm.set_tool_position(z = -5, speed = 50, wait = True)
    p1[2] = zprobeRel(1)[0]
    arm.set_tool_position(x = xd, y = yd, speed = 50, wait = True)
    p2[2] = zprobeRel(1)[0]
    arm.set_tool_position(y = -2*yd, speed = 50, wait = True)
    p3[2] = zprobeRel(1)[0]
    arm.set_tool_position(x = -xd, y = yd, speed = 50, wait = True) # move back to original position

    n = np.cross(p2 - p1, p3 - p1)
    nn = n / np.linalg.norm(n)
    angles = 180 * (np.arcsin(nn)) / np.pi
    arm.set_tool_position(roll = angles[1], pitch = -angles[0], speed = 50, wait = True)

    return angles[1], angles[0], p1, p2, p3

def lineupShelf(shelfCount):
    """
    Find position of build plates on shelf

    shelfCount: number of build plates on shelf
    """
    positions = np.zeros((shelfCount, 6))
    arm.set_mode(2)
    arm.set_state(0)
    arm.start_record_trajectory()
    input('Line up corner of tool with marker on top plate and press enter')
    arm.stop_record_trajectory()
    arm.set_mode(0)
    arm.set_state(0)
    arm.set_tool_position(x = -20, speed = 50, wait = True)
    arm.set_tool_position(z = -30, speed = 50, wait = True)
    arm.set_tool_position(x = 60, speed = 50, wait = True)
    zlevel()
    arm.set_tool_position(x = -60, speed = 50, wait = True)
    arm.set_tool_position(z = 17, speed = 50, wait = True)
    findYaw()
    # arm.set_tool_position(y = 30)
    findpoint()
    arm.set_tool_position(z = 30, speed = 50, wait = True)
    positions[0] = arm.get_position()[1]
    for i in range(1, len(positions)):
        input("next??")
        arm.set_tool_position(x = 30, speed = 50, wait = True)
        zlevel()
        arm.set_tool_position(x = -35, speed = 50, wait = True)
        arm.set_tool_position(z = 17, speed = 50, wait = True)
        findYaw()
        xprobe(1)
        arm.set_tool_position(z = 30, speed = 50, wait = True)
        positions[i] = arm.get_position()[1]
    return positions

def pickUp(position):
    """
    Pick up a build plate at a given location.

    position: position of build plate to be picked up
    """
    arm.set_position(*position, speed = 100, wait = True)
    #arm.set_tool_position(z = -2*np.cos(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(x = 40, speed = 100, wait = True)
    arm.set_tool_position(pitch = -6, speed = 100, wait = True)
    arm.set_tool_position(x = 42, z = -42*np.tan(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(z = -2, speed = 50, wait = True)
    arm.set_tool_position(pitch = 6, speed = 50, wait = True)
    arm.set_tcp_load(weight = 0.4997 + 0.048 + 1.609, center_of_gravity=[174,0,11])
    arm.set_tool_position(z = -20, speed = 10, wait = True)
    return arm.get_position()

def setDown(position):
    """
    Set down a build plate at a given location.

    position: position where build plate will be set down
    """
    arm.set_position(*(np.array(position) + [0, 0, (20 + 2*np.cos(6*np.pi/180)), 0, 0, 0]), speed = 100, wait = True)
    arm.set_tool_position(x = 40 + 45/np.cos(6*np.pi/180) + 2*np.tan(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(z = 20, speed = 10, wait = True)
    arm.set_tcp_load(weight = 0.4997 + 0.048, center_of_gravity=[41,0,18])
    arm.set_tool_position(pitch = -6, speed = 50, wait = True)
    arm.set_tool_position(z = 2, speed = 50, wait = True)
    arm.set_tool_position(x = -42, z = 42*np.tan(6*np.pi/180), speed = 100, wait = True)
    arm.set_tool_position(pitch = 6, speed = 100, wait = True)
    arm.set_tool_position(x = -40, speed = 100, wait = True)

def transferPlates(position1, position2, moveSpeed):
    """
    Transfer build plate between two positions.

    position1: pickup location of build plate
    position2: set down location of build plate
    moveSpeed: move speed of arm between locations
    """
    currentPosition = arm.get_position()[1]
    arm.set_tool_position(yaw = (currentPosition[5]-position1[5])/10, speed = moveSpeed, wait = True)
    pickUp(position1)
    arm.set_tool_position(x = -450, speed = moveSpeed, wait = True)
    arm.set_tool_position(yaw = (position1[5]-position2[5])/2, speed = moveSpeed, wait = True)
    arm.set_tool_position(yaw = (position1[5]-position2[5])/2, speed = moveSpeed, wait = True)
    arm.set_position(*(np.array(position2) + np.array([-350 * np.cos(position2[5] * np.pi / 180), -350 * np.sin(position2[5] * np.pi / 180), 20, 0, 0, 0])), speed = moveSpeed, wait = True)
    setDown(position2)

# Line up with shelf
Place the plate on the plate shelf. Then, manually align the side end stop to the height of the plate, positioning it about 10 mm to the left of the tape marker when facing the plate. After manual aligning, hit enter.

In [7]:
platePosition_shelf = lineupShelf(1) # line up with the plate on shelf to get shelf position

traveled_y: 0.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 1.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 1.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 2.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 2.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 3.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 3.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 4.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 4.5 x1: 5.0 x2: 4.0 delta: 1.0
Detected raised point / edge feature
detected at relative y: 4.5


# Line up with Jubilee
Place the plate on the Jubilee. Then, manually align the side end stop to the height of the plate, positioning it about 10 mm to the left of the tape marker when facing the plate. After manual aligning, hit enter.

In [6]:
platePosition_jubilee = lineupShelf(1)

[set_state], xArm is ready to move
traveled_y: 0.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 1.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 1.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 2.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 2.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 3.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 3.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 4.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 4.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 5.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 5.5 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 6.0 x1: 5.0 x2: 5.0 delta: 0.0
traveled_y: 6.5 x1: 5.0 x2: 4.0 delta: 1.0
Detected raised point / edge feature
detected at relative y: 6.5


# move plate 
pick up plate from platePosition_shelf[0] and set down on platePosition_jubilee[0].

In [10]:
transferPlates(platePosition_shelf[0], platePosition_jubilee[0], 100)

[SDK][ERROR][2026-06-23 17:43:01][base.py:168] - - [report-socket] socket read timeout
[SDK][ERROR][2026-06-23 17:47:34][base.py:271] - - [main-socket] socket read failed, len=0
[SDK][ERROR][2026-06-23 17:47:35][base.py:1342] - - report thread is break, connected=False, failed_cnts=14
